In [3]:
with open("data/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of characters:", len(raw_text))
print(raw_text[:99])

Total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


## Step 1 - Tokenization
### Testing use of `re` library for tokenization

Let's test usage of `re` library to split text into individual words

In [4]:
import re

text = "Hello world! This is a test-- of course. Let's see how it works."
sentences = re.split(r'(\s+)', text)

print(sentences)

['Hello', ' ', 'world!', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test--', ' ', 'of', ' ', 'course.', ' ', "Let's", ' ', 'see', ' ', 'how', ' ', 'it', ' ', 'works.']


consider puntuaction characters like `,` and `.` as tokens

In [5]:
sentences = re.split(r'([,.]|\s+)', text)
print(sentences)

['Hello', ' ', 'world!', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test--', ' ', 'of', ' ', 'course', '.', '', ' ', "Let's", ' ', 'see', ' ', 'how', ' ', 'it', ' ', 'works', '.', '']


consider additional puntuaction characters as tokens

In [6]:
sentences = re.split(r'([,.:;?_!"()\']|--|\s+)', text)
print(sentences)

['Hello', ' ', 'world', '!', '', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test', '--', '', ' ', 'of', ' ', 'course', '.', '', ' ', 'Let', "'", 's', ' ', 'see', ' ', 'how', ' ', 'it', ' ', 'works', '.', '']


remove blank tokens

In [7]:
sentences = [item for item in sentences if item.strip()]
print(sentences)

['Hello', 'world', '!', 'This', 'is', 'a', 'test', '--', 'of', 'course', '.', 'Let', "'", 's', 'see', 'how', 'it', 'works', '.']


### Use basic tokenization on entire text

In [8]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s+)', raw_text)
preprocessed = [item for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## Step 2 - Assign token IDs to tokens

### Unique set of tokens

Let's create a unique set of tokens.

In [9]:
## define unique tokens (list to set) and sort tokens
all_tokens = sorted(set(preprocessed))
print("Number of unique tokens:", len(all_tokens))

Number of unique tokens: 1130


## Vocabulary
Let's create a vocabulary mapping an ID to each token.

In [10]:
vocab = {token: idx for idx, token in enumerate(all_tokens)}


# for i, item in enumerate(all_tokens[:15]):
#     print(f"{i}: {item}")



## Tokenizer Class

In [23]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {idx: token for token, idx in vocab.items()}

    def tokenize(self, text):
        tokens = re.split(r'([,.:;?_!"()\']|--|\s+)', text)
        tokens = [item.strip() for item in tokens if item.strip()]
        return tokens

    def encode(self, text):
        tokens = self.tokenize(text)
        ids = [self.str_to_int[token] for token in tokens]
        return ids

    def decode(self, ids):
        tokens = [self.int_to_str[idx] for idx in ids]
        text = ' '.join(tokens)
        # remove spaces around punctuation
        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)
        return text

In [17]:
tokenizer = SimpleTokenizerV1(vocab)

text = """
"It's the last he painted, you know, Mrs. Gisburn said with pardonable pride."
"""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 67, 7, 38, 851, 1108, 754, 793, 7, 1]


In [18]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know, Mrs. Gisburn said with pardonable pride."'

## Adding special context tokes

The current version of tokenizer class does not handle tokens that are not part of the vocabulary. The code snippet here below will generate a runtime error since `Hello` is not a token in our vocabulary.

In [26]:
# text = "Hello, do you like tea?"
# print(tokenizer.encode(text))

The tokenizer and the vocabulary need changes to address this issue. 

Let's add special tokens `<|endoftext|>` and `<|unk|>` to the vocabulary.

In [27]:
all_tokens = sorted(set(preprocessed))
all_tokens.extend(['<|endoftext|>', '<|unk|>'])

vocab = {token: idx for idx, token in enumerate(all_tokens)}

In [28]:
len(vocab.items())

1132

In [30]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(f"{item}")

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [34]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {idx: token for token, idx in vocab.items()}

    def tokenize(self, text):
        tokens = re.split(r'([,.:;?_!"()\']|--|\s+)', text)
        tokens = [item.strip() for item in tokens if item.strip()]
        tokens = [token if token in self.str_to_int else '<|unk|>' for token in tokens]
        return tokens

    def encode(self, text):
        tokens = self.tokenize(text)
        ids = [self.str_to_int[token] for token in tokens]
        return ids

    def decode(self, ids):
        tokens = [self.int_to_str[idx] for idx in ids]
        text = ' '.join(tokens)
        # remove spaces around punctuation
        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)
        return text

In [35]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace, the guests were enjoying the afternoon tea."

text = " <|endoftext|> ".join([text1, text2])

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace, the guests were enjoying the afternoon tea.


In [37]:
print(tokenizer.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 5, 988, 1131, 1088, 1131, 988, 1131, 975, 7]


In [38]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>, the <|unk|> were <|unk|> the <|unk|> tea.
